
---

# Day 30: Functional Transformations & Preprocessing Pipelines

## 1. Quantile-Quantile (Q-Q) Plots Explained

A **Q-Q Plot** is a visual tool used to determine if a data column follows a specific distribution (typically a **Normal / Gaussian Distribution**).

* **The Red Line:** Represents a perfect theoretical normal distribution.
* **The Blue Dots:** Represent your actual data points sorted from lowest to highest.
* **How to Read It:**
* **Hugging the Line:** If the blue dots follow the straight red line closely, the feature is normally distributed (e.g., your `Age` column).
* **Curving Away / Vertical Spikes:** If the blue dots curve sharply off the line at the edges, the data is skewed and contains extreme outliers (e.g., your raw `Fare` column).



---

## 2. Mathematical Transformations & Their Best Uses

During this session, we evaluated 5 major functional transformations. Here is when and why to use them:

| Transformation | Mathematical Function | Best Used For | What It Did To Our Data |
| --- | --- | --- | --- |
| **Log Transform** | `np.log1p` $\rightarrow \log(x+1)$ | Highly Right-Skewed data with long tails (e.g., Income, Fares). | **Best overall.** Successfully compressed massive values and flattened the curve into a straight line. |
| **Square Root** | `np.sqrt` $\rightarrow \sqrt{x}$ | Moderately Right-Skewed data. | Milder than Log. It pulled outliers closer but wasn't as aggressive as the log transform. |
| **Reciprocal** | `1 / (x + 1)` | Datasets where tiny values/ratios are highly important. | **Caused convergence issues.** Flipped the data scale completely, making it harder for linear models to read. |
| **Square / Power** | `x ** 2` | Left-Skewed data (where values cluster at the high end). | Stretched out the large values even more. Not suitable for our right-skewed `Fare`. |
| **Sine Transform** | `np.sin` | Cyclical / Periodic data (e.g., hours of a day, months of a year). | **Destroyed performance.** Compressing wide ticket prices into a continuous wave waveform erased the continuous linear relationship. |

---

## 3. Scikit-Learn's `ColumnTransformer` Reordering Rule

When you pass specific columns into a `ColumnTransformer`, Scikit-Learn breaks the original Pandas DataFrame layout apart. The final output array follows one strict rule: **Transformation list order takes absolute priority over your original dataframe sequence.**

### The Array Zones:

1. **The Transformed Zone (Front):** Any column mentioned explicitly in a transformer tuple is processed and pushed to the very front in the exact order you listed them in your code.
2. **The Remainder Zone (Back):** If `remainder='passthrough'` is used, all unmentioned leftover columns are gathered up, **keeping their relative original order**, and glued to the absolute back of the matrix.

### Code Example:

```python
# Even though 'Fare' is the last column in the original DataFrame...
trf = ColumnTransformer([
    ('log', FunctionTransformer(np.log1p), ['Fare'])
], remainder='passthrough')

# In the output matrix:
# Index 0 = Transformed 'Fare'
# Index 1 = Leftover 'Age' (Passthrough)

```

---

## 4. Key Implementation Pitfalls Avoided

### 1. Missing `random_state` in Splitting

* **The Problem:** Running `train_test_split(X, y)` without a seed causes Scikit-Learn to shuffle data completely at random on every cell execution.
* **The Consequence:** Baseline baseline accuracies will bounce randomly (e.g., `0.65` to `0.69`), making it impossible to tell if your transformation actually improved performance or if you just got a "lucky" data split.
* **The Fix:** Always specify `random_state` to lock down reproducibility:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

```



### 2. Logistic Regression Convergence Warnings

* **The Problem:** Modifying distributions can introduce wide structural scales that cause optimization algorithms (like the default `lbfgs` solver) to get mathematically stuck. It runs out of its default limit of 100 steps.
* **The Fix:** Grant the optimizer more steps/breathing room to converge on the ideal weights:
```python
clf = LogisticRegression(max_iter=1000)

```



---

## 5. Summary Insights on Model Behavior

* **Decision Trees are invariant to functional transforms:** Tree models separate classes by making threshold cuts (e.g., `Is Fare > 50?`). Transforming data monotonically simply shifts the questions threshold proportionately (`Is Log(Fare) > Log(50)?`) without changing which rows fall into which branch. Accuracy remains flat.
* **Linear Models rely on normal distributions:** Logistic Regression relies heavily on feature geography. Straightening out a skewed feature using a log transform helps the model calculate a more reliable decision boundary line, resulting in an accuracy boost.